In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
llm=ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

In [4]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [6]:
def generate_joke(state:JokeState)->dict:
    prompt=f"generate a joke on the topic {state['topic']}"
    response=llm.invoke(prompt).content

    return {'joke':response}


In [7]:
def generate_explanation(state:JokeState)->dict:
    prompt=f"Write an explanation for the following joke {state['joke']}"
    response=llm.invoke(prompt).content

    return {'explanation':response}


In [8]:
graph=StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)


In [9]:
config1={"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.',
   'extras': {'signature': 'EmAKXgFpFH0T/0/cf34yWcieiDn9saWg1XPUpiYGE4IDC8FBPCUQZ2FljFo8bvERHXWrjkCanyk2u4L8/FsSB1/hFSaSjVDYEtGckfkCWr1hbi7Ikcq7T29Cu6AUqIC2+aE='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke is a pun that plays on a common stereotype about hipsters and a popular phrase used when describing pizza.\n\nHere is the breakdown:\n\n1. **The Stereotype:** Hipsters are often stereotyped as being obsessed with being trendy, underground, and liking things *before* they become mainstream or popular ("cool"). \n2. **The Literal Meaning:** When referring to food like pizza, people often say "wait until it\'s cool" meaning you should let it cool down so you don\'t burn your mouth. \n\n**The Punchline:** The hipster burned his mouth because he ate the pizza *literally* before it had time to cool down, but the joke uses the 

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0T/0/cf34yWcieiDn9saWg1XPUpiYGE4IDC8FBPCUQZ2FljFo8bvERHXWrjkCanyk2u4L8/FsSB1/hFSaSjVDYEtGckfkCWr1hbi7Ikcq7T29Cu6AUqIC2+aE='}}], 'explanation': [{'type': 'text', 'text': 'This joke is a pun that plays on a common stereotype about hipsters and a popular phrase used when describing pizza.\n\nHere is the breakdown:\n\n1. **The Stereotype:** Hipsters are often stereotyped as being obsessed with being trendy, underground, and liking things *before* they become mainstream or popular ("cool"). \n2. **The Literal Meaning:** When referring to food like pizza, people often say "wait until it\'s cool" meaning you should let it cool down so you don\'t burn your mouth. \n\n**The Punchline:** The hipster burned his mouth because he ate the pizza *literally* before it had time to cool down, but the jok